# Dimension-derived Variable Construction (English-only)

This notebook derives analysis-ready variables from the unified Dimensions dataset and writes a single Parquet for downstream analyses. The workflow adheres to academic reporting best practices: each variable is introduced by a short rationale (markdown) followed by a reproducible code cell. No intermediate files are written; only the final dataset is saved.

- Source dataset: `$DATA_ROOT/processed/dimension_merged.parquet`
- External reference (for disciplinary core journal matching): `$DATA_ROOT/processed/scimagojr_communication_journal_1999_2024.csv`
- Output dataset: `$DATA_ROOT/processed/dimension_data_for_analysis.parquet`

Conventions:
- All module outputs include `id` for reliable merges.
- Column blocks are ordered by conceptual variable groups so related columns appear together in the final file.
- Time-window logic for invisibility is intentionally not applied here; analyses may add temporal filters later.


In [ ]:
%run research/dimensions-dataset-construction/analysis/create_variables.py


## Schema preview

We first inspect the Parquet schema and a small sample to confirm column availability and obtain a quick sense of the data. No full-table prints are performed to avoid large outputs.


## Variable: Invisibility (times_cited only)

Definition: A record is labeled as invisible if it has received zero citations, i.e., `invisibility = 1` when `times_cited == 0`; otherwise `0`. We retain `date` for analytical context but do not use it in the rule here. All outputs include `id` for merging.

Columns used: `id`, `times_cited`, `date`.

Output columns: `id`, `invisibility`, `times_cited`, `date`. 


## Variable: Geographic (institutional location)

We retain institutional location fields to support geographic analyses. No transformation is applied here; the values may contain multiple institutions and countries per record.

Columns used: `id`, `research_org_country_names`, `research_org_names`, `research_org_types`.

Output columns: `id`, `research_org_country_names`, `research_org_names`, `research_org_types`. 


## Variable: Topical (concepts)

We retain topic-related fields to enable later construction of mainstream-topic shares or binary mainstream indicators. No transformation is performed here.

Columns used: `id`, `concepts`, `concepts_scores`.

Output columns: `id`, `concepts`, `concepts_scores`. 


## Variable: Disciplinary (core journal hit via SJR)

We construct a binary indicator for whether the record's journal identifiers intersect the SJR communication journals ISSN list. The variable equals 1 if any standardized token from `issn` or `isbn` matches any SJR `Issn`; otherwise 0.

Columns used: `id`, `issn`, `isbn` (from Dimensions) and `Issn` (from SJR CSV).

Output columns: `id`, `issn`, `isbn`, `disciplinary`.

Normalization policy:
- Uppercase tokens; drop hyphens/whitespace; retain only digits and `X`.
- Support multi-valued identifiers split by common non-alphanumerics.


## Variable: Prestige (institutional ranking category)

We derive a prestige category by fuzzy-matching `research_org_names` against SCImago Institution Rankings (communication), then mapping rank to bins:
- Elite: top 100
- High: 101–500
- Medium: 501–1000
- Low: Unranked or no confident match

Inputs:
- `research_org_names` (may contain multiple institutions per record)
- Rankings CSV: `$DATA_ROOT/processed/scimagoir_2025_Overall Rank_Communication.csv` (expects columns like `institution` and an overall-rank column)

Output columns: `id`, `prestige`. 


## Variable: Open Access status

We retain the open access indicator as-is for subsequent stratified analyses.

Columns used: `id`, `open_access`.

Output columns: `id`, `open_access`. 


## Variable: First author experience

Definition: The difference between this paper's publication year and the earliest year in which the same author is listed first on a publication observed within this dataset.

- Identification of first author: prefer stable IDs; we do not use name-based matching to avoid collisions
  - Priority: researchers[0] → authors[0]; accepts a stable ID in direct or nested ID metadata, with no name-based fallback
- Year source: prefer `year`; fallback to extracting the year from `date`
- Output columns: `id`, `first_author_experience`
- Debug-only (not merged into final output): `first_author_key`, `first_author_first_year`


## Variables: Controls (document and referencing)

We retain common control variables for modeling and descriptive statistics.

Columns used: `id`, `document_type`, `type`, `authors_count`, `reference_ids`, `referenced_pubs`.

Output columns: `id`, `document_type`, `type`, `authors_count`, `reference_ids`, `referenced_pubs`. 


## Merge and write output (column order by conceptual blocks)

We left-join all module dataframes by `id` and order columns so that variables belonging to the same construct appear together. The final dataset is written as a single Parquet file with Snappy compression.


## Data quality and reliability checks

We perform lightweight but informative reliability checks on the final dataset:
- Missingness summary (counts and ratios) at column level
- Key constraints: `id` uniqueness and row-count parity against the base table
- Domain checks: `invisibility` in {0,1}; non-negativity for `times_cited` and `authors_count`
- Format checks: ISSN/ISBN token validity (normalized tokens of length 8 with digits/X)
- Distribution snapshots: value counts or quantiles for selected variables
- Risk indicators: normalized duplicate DOI (if a DOI column exists upstream), rows missing both `issn` and `isbn`
